# FNO · альтернативная регрессионная голова

Эксперимент `fno_less_augmented`. Индексы — `data/splits/`; изображения и метки — `data/processed/`. Выходные массивы сохраняются в `results/fno_less/normalized_augmented/` с именами `losses.npy`, `targets.npy`, `predictions.npy`.

Архитектура и параметры исходного обучения сохранены. Пути определяются автоматически, папка результатов создаётся при сохранении. См. [воспроизведение](../../docs/reproduction.md) и [результаты](../../docs/results.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Импорты и путь к проекту


In [ ]:
import numpy as np
import sys
import os
import time
import random
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader


from galaxy_mass_estimation.models.fno import ModelFNO2
from galaxy_mass_estimation.data import ACTClusterDataset

## Воспроизводимость


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)

## Устройство


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Параметры обучения


In [ ]:
EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 128

## Модель, оптимизатор и функция потерь


In [ ]:
model = ModelFNO2().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = torch.nn.MSELoss()

## Индексы выборок


In [ ]:
train_indices = np.load(PROJECT_ROOT / "data/splits/augmented/train_indices.npy")
test_indices = np.load(PROJECT_ROOT / "data/splits/augmented/test_indices.npy")

## Датасеты


In [ ]:
train_dataset = ACTClusterDataset(
    str(PROJECT_ROOT / "data/processed"),
    train_indices,
    norm=True,
    augm=True,
)
test_dataset = ACTClusterDataset(
    str(PROJECT_ROOT / "data/processed"),
    test_indices,
    norm=True,
    augm=True,
)

## Загрузчики данных


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

## Обучение и промежуточная оценка


In [ ]:
model.train()
losses = []

for epoch in range(EPOCHS):
    start_t = time.time()
    total_loss = 0
    for img, scalars, target in train_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        pred = model(img, scalars)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(train_loader))

    scheduler.step()

    if epoch == 0:
        print("Epoch 1 time:", time.time() - start_t)

    # --- оценка каждые 10 эпох ---
    if (epoch + 1) % 10 == 0:
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for img, scalars, target in test_loader:
                img = img.to(device)
                scalars = scalars.to(device)
                target = target.to(device)

                pred = model(img, scalars)
                preds.append(pred)
                targets.append(target)

        model.train()

        preds = torch.cat(preds)
        targets = torch.cat(targets)

        mae = torch.abs(targets - preds)
        mean_mae = mae.mean().item()
        max_mae = mae.max().item()

        relative_errors = mae / targets
        mean_rel_error = relative_errors.mean().item()
        max_rel_error = relative_errors.max().item()

        print(f"EPOCH={epoch+1}")
        print(f"Средняя MAE {mean_mae:.4f}")
        print(f"Максимальная MAE {max_mae:.4f}")
        print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
        print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")
losses = np.asarray(losses)

Epoch 1 time: 29.425983667373657
EPOCH=10
Средняя MAE 0.1602
Максимальная MAE 0.8337
Средняя относительная ошибка: 0.2891
Максимальная относительная ошибка: 1.6764
EPOCH=20
Средняя MAE 0.1556
Максимальная MAE 0.7469
Средняя относительная ошибка: 0.2908
Максимальная относительная ошибка: 1.4807
EPOCH=30
Средняя MAE 0.0840
Максимальная MAE 0.6314
Средняя относительная ошибка: 0.1485
Максимальная относительная ошибка: 1.2357
EPOCH=40
Средняя MAE 0.0895
Максимальная MAE 0.4684
Средняя относительная ошибка: 0.1592
Максимальная относительная ошибка: 1.2574
EPOCH=50
Средняя MAE 0.0897
Максимальная MAE 0.5814
Средняя относительная ошибка: 0.1596
Максимальная относительная ошибка: 1.2640
EPOCH=60
Средняя MAE 0.0911
Максимальная MAE 0.6668
Средняя относительная ошибка: 0.1600
Максимальная относительная ошибка: 1.1589
EPOCH=70
Средняя MAE 0.0922
Максимальная MAE 0.6902
Средняя относительная ошибка: 0.1607
Максимальная относительная ошибка: 1.0768
EPOCH=80
Средняя MAE 0.0931
Максимальная MAE 0.707

## Итоговые предсказания


In [ ]:
model.eval()  # режим оценки
preds = []
targets = []

with torch.no_grad():
    for img, scalars, target in test_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        pred = model(img, scalars)

        preds.append(pred.cpu())
        targets.append(target.cpu())

preds = torch.cat(preds)
targets = torch.cat(targets)

mae = torch.abs(targets - preds)
mean_mae = mae.mean().item()
max_mae = mae.max().item()

relative_errors = mae / torch.abs(targets)
mean_rel_error = relative_errors.mean().item()
max_rel_error = relative_errors.max().item()

print(f"EPOCH={epoch+1}")
print(f"Средняя MAE: {mean_mae:.4f}")
print(f"Максимальная MAE: {max_mae:.4f}")
print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

EPOCH=100
Средняя MAE: 0.0934
Максимальная MAE: 0.6951
Средняя относительная ошибка: 0.1617
Максимальная относительная ошибка: 1.0871


## Сохранение результатов


In [ ]:
output_dir = PROJECT_ROOT / "results/fno_less/normalized_augmented"
output_dir.mkdir(parents=True, exist_ok=True)
np.save(output_dir / "losses.npy", losses)
np.save(output_dir / "targets.npy", targets)
np.save(output_dir / "predictions.npy", preds)